# 16 - Purpose-bound Agent Data Windows

One agent-access requirement has four variables: database,
purpose, window anchor, and lookback length. This notebook makes
all four combinations explicit instead of hiding them behind one
generic retention rule:

| Database | Purpose | Window |
| --- | --- | --- |
| `master` | research | rolling 90 days |
| `master` | commercial | rolling 30 days |
| `product` | research | as-of 90 days |
| `product` | commercial | as-of 30 days |

Rolling windows are anchored to the server evaluation time. As-of
windows are reproducible: the caller must state the anchor in
`data_access_context.as_of`. Authorization returns the required
bounds; query validation proves SQL is no broader than them.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the Customer 360 demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


PRODUCT_DATABASE_TABLES = {"product_usage_events", "support_tickets", "ml_feature_store"}


def asset(table, column=None, schema="public", database=None):
    resolved_database = database or (
        "product" if table in PRODUCT_DATABASE_TABLES else "master"
    )
    ref = {"database": resolved_database, "schema": schema, "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


In [ ]:
# The access-window policies are bound to the exact `agent`
# role. Offline mode replays those recordings; live mode uses
# a separate agent-bound token so the rest of the pack can
# retain its identity-neutral release credential.
agent_client = get_client(token_env="METATATE_SAAS_MCP_AGENT_TOKEN")


## 1. Authorize every combination


In [ ]:
AS_OF = "2026-08-01T00:00:00Z"

authorization_cases = [
    ("master / research", dict(
        asset=asset("customers"),
        use="research recent customer behavior",
        scenario_key="access.read",
        purpose_key="research.general",
    )),
    ("master / commercial", dict(
        asset=asset("customers"),
        use="prepare a commercial customer analysis",
        scenario_key="access.read",
        purpose_key="commercial.general",
    )),
    ("product / research", dict(
        asset=asset("product_usage_events", database="product"),
        use="reproduce product research as of a fixed date",
        scenario_key="access.read",
        purpose_key="research.general",
        data_access_context={"as_of": AS_OF},
    )),
    ("product / commercial", dict(
        asset=asset("product_usage_events", database="product"),
        use="reproduce a commercial product analysis as of a fixed date",
        scenario_key="access.read",
        purpose_key="commercial.general",
        data_access_context={"as_of": AS_OF},
    )),
]

authorizations = {}
for label, arguments in authorization_cases:
    answer = agent_client.authorize_use(**arguments)
    authorizations[label] = answer
    window = next(
        (c for c in answer.get("conditions", [])
         if c.get("kind") == "data_window_required"),
        {},
    )
    projection = window.get("projection") or {}
    print(
        f"{label:22} -> {answer_label(answer):12} "
        f"{projection.get('type', '?'):7} "
        f"{projection.get('lookback_days', '?')} days"
    )


The decision is conditional, not an unconditional allow. The
condition is executable evidence: it names the time column and
exact lower/upper bounds an agent must apply before reading data.


## 2. Missing context fails closed


In [ ]:
missing_purpose = agent_client.authorize_use(
    asset("customers"),
    use="read recent customer records",
    scenario_key="access.read",
)
missing_as_of = agent_client.authorize_use(
    asset("product_usage_events", database="product"),
    use="read product events for research",
    scenario_key="access.read",
    purpose_key="research.general",
)

print("missing purpose ->", missing_purpose["state"], missing_purpose["reason_code"])
print("missing as_of   ->", missing_as_of["state"], missing_as_of["reason_code"])


Metatate does not infer research versus commercial use, and it
does not substitute the current time for a missing as-of anchor.
Both omissions return a typed review requirement.


## 3. Prove the SQL stays inside the authorized window


In [ ]:
validation_cases = [
    ("master research 90", dict(
        sql="SELECT customer_id, account_status FROM master.public.customers WHERE created_at >= CURRENT_TIMESTAMP - INTERVAL '90 days' AND created_at <= CURRENT_TIMESTAMP",
        scenario_key="access.read", default_database="master", default_schema="public",
        purpose_key="research.general",
    )),
    ("master commercial 30", dict(
        sql="SELECT customer_id, account_status FROM master.public.customers WHERE created_at >= CURRENT_TIMESTAMP - INTERVAL '30 days' AND created_at <= CURRENT_TIMESTAMP",
        scenario_key="access.read", default_database="master", default_schema="public",
        purpose_key="commercial.general",
    )),
    ("product research 90", dict(
        sql="SELECT customer_id, event_name FROM product.public.product_usage_events WHERE occurred_at >= TIMESTAMPTZ '2026-05-03T00:00:00Z' AND occurred_at <= TIMESTAMPTZ '2026-08-01T00:00:00Z'",
        scenario_key="access.read", default_database="product", default_schema="public",
        purpose_key="research.general", data_access_context={"as_of": AS_OF},
    )),
    ("product commercial 30", dict(
        sql="SELECT customer_id, event_name FROM product.public.product_usage_events WHERE occurred_at >= TIMESTAMPTZ '2026-07-02T00:00:00Z' AND occurred_at <= TIMESTAMPTZ '2026-08-01T00:00:00Z'",
        scenario_key="access.read", default_database="product", default_schema="public",
        purpose_key="commercial.general", data_access_context={"as_of": AS_OF},
    )),
]

for label, arguments in validation_cases:
    answer = agent_client.validate_query_context(**arguments)
    print(f"{label:24} -> {answer.get('verdict')} / {answer.get('state')}")


## 4. A wider window is rejected


In [ ]:
broad_rolling = agent_client.validate_query_context(
    "SELECT customer_id, account_status FROM master.public.customers WHERE created_at >= CURRENT_TIMESTAMP - INTERVAL '90 days' AND created_at <= CURRENT_TIMESTAMP",
    scenario_key="access.read", default_database="master", default_schema="public",
    purpose_key="commercial.general",
)
broad_as_of = agent_client.validate_query_context(
    "SELECT customer_id, event_name FROM product.public.product_usage_events WHERE occurred_at >= TIMESTAMPTZ '2026-05-03T00:00:00Z' AND occurred_at <= TIMESTAMPTZ '2026-08-01T00:00:00Z'",
    scenario_key="access.read", default_database="product", default_schema="public",
    purpose_key="commercial.general", data_access_context={"as_of": AS_OF},
)

print("commercial rolling 90 ->", broad_rolling.get("verdict"))
print("commercial as-of 90   ->", broad_as_of.get("verdict"))


The commercial policy permits 30 days. Asking for 90 days is a
broader read and fails, even though the same SQL shape is valid
for research. Database, purpose, anchor type, and duration all
remain decision-bearing inputs.
